# Does training noise change what the network learns? — v8

The hypothesis under test is that the **baseline noise in the stimuli is what pushes the network to spread
both tasks across both hidden units**, producing the rotated solution rather than a clean one-unit-per-task
split. Removing the noise entirely (0.1 versus 0.0) barely changed anything, but that is a narrow test: at
a stimulus intensity drawn from 1 to 3, a noise standard deviation of 0.1 is a very small perturbation, so
the two conditions being compared were both effectively "low noise". A fair test needs noise that is
**comparable in scale to the signal**.

This notebook therefore **trains** networks across a wide range of noise levels, from none up to noise as
large as the stimulus itself, and asks two separate questions:

1. **Performance.** How does accuracy fall as training noise rises, and where does the task become
   impossible?
2. **Geometry.** Does the *solution the network finds* change with noise? This is measured rather than judged by eye, using the task-aligned decoder axes: the angle between the *where*
   and *whether* axes (is the code factorised?) and their alignment to the hidden-unit axes (is each task
   carried by its own unit, or are both shared?).

If the noise hypothesis is right, the alignment should move systematically with training noise: low noise
should give axis-aligned solutions (near 0 degrees) and high noise rotated ones (near 45 degrees). If
alignment is flat across a wide noise range, the hypothesis is refuted properly rather than provisionally.

Requires `scikit-learn`. Trials are generated in-notebook so the training noise can be set freely.

## 1. Setup

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from pathlib import Path

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cpu")
OUT_DIR = Path("./noise_training_v8"); OUT_DIR.mkdir(exist_ok=True)
T = 50; T_ON, T_OFF = 10, 20
CLASS_COLORS = ["#cfcfcf", "#e0852e", "#c0392b", "#2e6ca4"]

## 2. Trial generator with a noise knob

The v8 specification with the baseline noise standard deviation as a parameter. Note the scale: stimulus
intensity is drawn uniformly from 1.0 to 3.0, so a noise standard deviation of 1.0 is already comparable
to a weak stimulus and 2.0 exceeds most of them. That is the range where an effect, if there is one,
should appear.

In [ ]:
SUBTASKS = ["det_absent","det_auditory_only","det_visual_only",
            "loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R","det_multisensory",
            "loc_conflict_audL_visR","loc_conflict_audR_visL"]
CONFLICT = ["det_multisensory","loc_conflict_audL_visR","loc_conflict_audR_visL"]
NONCONF  = [s for s in SUBTASKS if s not in CONFLICT]
LOC_FIT  = ["loc_auditory_only_L","loc_auditory_only_R","loc_visual_only_L","loc_visual_only_R",
            "loc_multisensory_same_L","loc_multisensory_same_R"]
DET_FIT  = ["det_absent","det_auditory_only","det_visual_only"]
TRAIN_COUNTS = {"det_absent":600,"det_visual_only":600,"det_auditory_only":1200,
                "loc_auditory_only_L":300,"loc_auditory_only_R":300,"loc_visual_only_L":300,
                "loc_visual_only_R":300,"loc_multisensory_same_L":300,"loc_multisensory_same_R":300,
                "loc_conflict_audL_visR":300,"loc_conflict_audR_visL":300}

def make_trial(sub, noise_std, rng, split):
    X = rng.normal(0.0, noise_std, (4, T)).astype(np.float32) if noise_std > 0 else np.zeros((4, T), np.float32)
    def add(ch, i): X[ch, T_ON:T_OFF] += i
    ai = vi = np.nan
    if   sub == "det_absent": lab = 0
    elif sub == "det_auditory_only": i = rng.uniform(1,3); add(0,i); add(1,i); lab = 1
    elif sub == "det_visual_only":   i = rng.uniform(1,3); add(2,i); add(3,i); lab = 0
    elif sub == "loc_auditory_only_L": add(0, rng.uniform(1,3)); lab = 3
    elif sub == "loc_auditory_only_R": add(1, rng.uniform(1,3)); lab = 2
    elif sub == "loc_visual_only_L":   add(2, rng.uniform(1,3)); lab = 3
    elif sub == "loc_visual_only_R":   add(3, rng.uniform(1,3)); lab = 2
    elif sub == "loc_multisensory_same_L": i = rng.uniform(1,3); add(0,i); add(2,i); lab = 3
    elif sub == "loc_multisensory_same_R": i = rng.uniform(1,3); add(1,i); add(3,i); lab = 2
    elif sub == "det_multisensory": i = rng.uniform(1,3); [add(c,i) for c in range(4)]; lab = 1
    elif sub == "loc_conflict_audL_visR":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(0, ai); add(3, vi)
        lab = (3 if ai > vi else 2) if split == "test" else int(rng.integers(2,4))
    elif sub == "loc_conflict_audR_visL":
        ai = rng.uniform(1,3); vi = rng.uniform(1,3); add(1, ai); add(2, vi)
        lab = (2 if ai > vi else 3) if split == "test" else int(rng.integers(2,4))
    return X, int(lab), np.float32(ai), np.float32(vi)

def make_split(split, noise_std, seed=0):
    rng = np.random.default_rng(1000 + seed + (0 if split == "train" else 7))
    Xs, ys, ts, ais, vis = [], [], [], [], []
    counts = TRAIN_COUNTS if split == "train" else {s: 100 for s in SUBTASKS}
    for sub, n in counts.items():
        for _ in range(n):
            X, lab, ai, vi = make_trial(sub, noise_std, rng, split)
            Xs.append(X); ys.append(lab); ts.append(sub); ais.append(ai); vis.append(vi)
    return {"X": np.stack(Xs), "y": np.array(ys, np.int64), "types": np.array(ts),
            "aud_int": np.array(ais, np.float32), "vis_int": np.array(vis, np.float32)}

## 3. Models (standard and masked)

In [ ]:
class StandardGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden
        self.gru = nn.GRU(n_in, hidden, batch_first=True); self.readout = nn.Linear(hidden, n_out)
    def forward(self, x):
        x = x.transpose(1, 2); h, _ = self.gru(x); return self.readout(h)
    def hidden_states(self, X):
        self.eval()
        with torch.no_grad(): h, _ = self.gru(torch.from_numpy(X).transpose(1, 2))
        return h.numpy()

class MaskedGRU(nn.Module):
    def __init__(self, n_in=4, hidden=2, n_out=4):
        super().__init__(); self.H = hidden; s = 1.0/math.sqrt(hidden)
        self.weight_ih = nn.Parameter(torch.empty(3*hidden, n_in).uniform_(-s, s))
        self.weight_hh = nn.Parameter(torch.empty(3*hidden, hidden).uniform_(-s, s))
        self.bias_ih   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.bias_hh   = nn.Parameter(torch.empty(3*hidden).uniform_(-s, s))
        self.readout   = nn.Linear(hidden, n_out)
        self.register_buffer("mask", torch.eye(hidden).repeat(3, 1))
    def masked_hh(self): return self.weight_hh * self.mask
    def _run(self, x):
        B, Tt, _ = x.shape; H = self.H; Whh = self.masked_hh()
        Wir, Wiz, Win = self.weight_ih[:H], self.weight_ih[H:2*H], self.weight_ih[2*H:]
        Whr, Whz, Whn = Whh[:H], Whh[H:2*H], Whh[2*H:]
        bir, biz, bin_ = self.bias_ih[:H], self.bias_ih[H:2*H], self.bias_ih[2*H:]
        bhr, bhz, bhn = self.bias_hh[:H], self.bias_hh[H:2*H], self.bias_hh[2*H:]
        h = x.new_zeros(B, H); hs = []
        for t in range(Tt):
            xt = x[:, t, :]
            r = torch.sigmoid(xt @ Wir.T + bir + h @ Whr.T + bhr)
            z = torch.sigmoid(xt @ Wiz.T + biz + h @ Whz.T + bhz)
            n = torch.tanh(xt @ Win.T + bin_ + r * (h @ Whn.T + bhn))
            h = (1 - z) * n + z * h; hs.append(h)
        return torch.stack(hs, 1)
    def forward(self, x): return self.readout(self._run(x.transpose(1, 2)))
    def hidden_states(self, X):
        self.eval()
        with torch.no_grad(): h = self._run(torch.from_numpy(X).transpose(1, 2))
        return h.numpy()

## 4. Configuration

The noise levels span from none to twice the weakest stimulus, so the sweep covers the regime where noise
is negligible through to the regime where it dominates. `KINDS` lets the masked and standard models be
compared on the same axes.

In [ ]:
HIDDEN = 2
KINDS  = ["masked", "standard"]           # drop one to halve the runtime
SEEDS  = [0, 1, 2, 3, 4]
TRAIN_NOISE_LEVELS = [0.0, 0.1, 0.25, 0.5, 1.0, 1.5, 2.0]
N_EPOCHS = 50; LR = 1e-3; BATCH = 64
FIT_FROM = T_ON                            # timesteps used for decoder fitting
print("models to train:", len(KINDS)*len(TRAIN_NOISE_LEVELS)*len(SEEDS))
print("note: stimulus intensity is U[1,3], so noise 1.0 is comparable to a weak stimulus")

## 5. Train, evaluate, and measure the geometry

In [ ]:
def train_model(kind, train_set, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = (MaskedGRU if kind == "masked" else StandardGRU)(4, HIDDEN, 4).to(device)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_set["X"]), torch.from_numpy(train_set["y"])),
                        batch_size=BATCH, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=LR); loss_fn = nn.CrossEntropyLoss()
    for _ in range(N_EPOCHS):
        model.train()
        for Xb, yb in loader:
            lo = model(Xb); B, Tt, C = lo.shape
            loss = loss_fn(lo.reshape(B*Tt, C), yb.unsqueeze(1).expand(B, Tt).reshape(B*Tt))
            opt.zero_grad(); loss.backward(); opt.step()
    model.eval(); return model

def evaluate(model, test_set):
    model.eval()
    with torch.no_grad(): pred = model(torch.from_numpy(test_set["X"]))[:, -1, :].argmax(-1).numpy()
    ty, y = test_set["types"], test_set["y"]
    accs = {s: float((pred[ty==s]==y[ty==s]).mean()) for s in SUBTASKS}
    return float(np.mean([accs[s] for s in NONCONF])), accs

def fit_task_axes(model, test_set):
    """where/whether decoder axes, fit on non-conflict trials only, signs pinned, unit-normalised."""
    Hs = model.hidden_states(test_set["X"]); tt = np.arange(T) >= FIT_FROM
    ty, y4 = test_set["types"], test_set["y"]
    def build(mask_trials, pos):
        idx = np.where(mask_trials)[0]
        Xf = Hs[idx][:, tt, :].reshape(-1, Hs.shape[2])
        yf = np.repeat((y4[idx] == pos).astype(int), tt.sum())
        return Xf, yf
    Xw, yw = build(np.isin(ty, LOC_FIT), 2)      # right vs left
    Xd, yd = build(np.isin(ty, DET_FIT), 1)      # det vs no det
    out = {}
    for name, (Xf, yf) in [("where", (Xw, yw)), ("whether", (Xd, yd))]:
        if len(np.unique(yf)) < 2:
            out[name] = (np.full(Hs.shape[2], np.nan), np.nan); continue
        dec = LogisticRegression(max_iter=2000).fit(Xf, yf)
        w = dec.coef_[0].astype(np.float64)
        if (Xf[yf == 1] @ w).mean() < (Xf[yf == 0] @ w).mean(): w = -w      # pin sign
        acc = float(dec.score(Xf, yf))
        out[name] = (w / (np.linalg.norm(w) + 1e-12), acc)
    w_where, acc_w = out["where"]; w_whether, acc_d = out["whether"]
    ang = float(np.degrees(np.arccos(np.clip(abs(w_where @ w_whether), 0, 1))))
    return dict(w_where=w_where, w_whether=w_whether, angle=ang, acc_where=acc_w, acc_whether=acc_d)

def alignment_to_units(w):
    """Angle from the nearest hidden-unit axis: 0 = one unit per task, 45 = fully shared (2D only)."""
    if np.any(np.isnan(w)): return np.nan
    a = np.degrees(np.arctan2(abs(w[1]), abs(w[0])))
    return float(min(a, 90 - a))

## 6. Run the sweep

For every training noise level the network is trained on data at that noise, then evaluated two ways: on a
**matched** test set at the same noise (how well it does in the world it was trained for) and on a **clean**
test set (how much of the task it has actually learned, with the noise taken away). The geometry is
measured on the matched test set.

In [ ]:
test_clean = make_split("test", 0.0)
test_by_noise = {ns: make_split("test", ns) for ns in TRAIN_NOISE_LEVELS}

res = {k: {ns: {"acc_matched": [], "acc_clean": [], "angle": [], "align_where": [], "align_whether": [],
                "acc_dec_where": [], "acc_dec_whether": []}
           for ns in TRAIN_NOISE_LEVELS} for k in KINDS}
example_models = {}
t0 = time.time()
for kind in KINDS:
    for ns in TRAIN_NOISE_LEVELS:
        tr = make_split("train", ns)
        for seed in SEEDS:
            m = train_model(kind, tr, seed)
            am, _ = evaluate(m, test_by_noise[ns])
            ac, _ = evaluate(m, test_clean)
            ax = fit_task_axes(m, test_by_noise[ns])
            r = res[kind][ns]
            r["acc_matched"].append(am); r["acc_clean"].append(ac)
            r["angle"].append(ax["angle"])
            r["align_where"].append(alignment_to_units(ax["w_where"]))
            r["align_whether"].append(alignment_to_units(ax["w_whether"]))
            r["acc_dec_where"].append(ax["acc_where"]); r["acc_dec_whether"].append(ax["acc_whether"])
            if seed == SEEDS[0]: example_models[(kind, ns)] = (m, ax)
        print("%-9s noise %.2f: matched %.3f  clean %.3f  axes %.1f deg  align(where) %.1f deg"
              % (kind, ns, np.mean(res[kind][ns]["acc_matched"]), np.mean(res[kind][ns]["acc_clean"]),
                 np.nanmean(res[kind][ns]["angle"]), np.nanmean(res[kind][ns]["align_where"])), flush=True)
print("total %.1f min" % ((time.time()-t0)/60))

## Figure 1 — performance against training noise

Two curves per model. **Matched** is accuracy on a test set at the same noise the network trained on, so
it measures how solvable the task is at that noise level. **Clean** removes the noise at
test time and asks how much of the underlying task the network learned regardless.

In [ ]:
kind_style = {"masked": ("tab:red", "s"), "standard": ("black", "o")}
fig, axs = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, key, name in [(axs[0], "acc_matched", "tested at the training noise (matched)"),
                      (axs[1], "acc_clean", "tested on clean stimuli")]:
    for kind in KINDS:
        m = np.array([np.mean(res[kind][ns][key]) for ns in TRAIN_NOISE_LEVELS])
        sd = np.array([np.std(res[kind][ns][key]) for ns in TRAIN_NOISE_LEVELS])
        c, mk = kind_style[kind]
        ax.errorbar(TRAIN_NOISE_LEVELS, m, yerr=sd, marker=mk, lw=2.0, capsize=3, color=c, label=kind)
        ax.fill_between(TRAIN_NOISE_LEVELS, m-sd, m+sd, color=c, alpha=0.12)
    ax.axhline(0.25, color="gray", ls=":", alpha=0.7, label="4-class chance")
    ax.axvline(0.1, color="steelblue", ls="--", alpha=0.5)
    ax.text(0.12, 0.03, "original noise (0.1)", color="steelblue", fontsize=8, rotation=90, va="bottom")
    ax.set_xlabel("training noise (baseline standard deviation)"); ax.set_ylabel("non-conflict accuracy")
    ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.set_title(name)
axs[0].legend(loc="lower left", fontsize=9, frameon=False)
fig.suptitle("Effect of training noise on performance (%d hidden units, %d seeds)" % (HIDDEN, len(SEEDS)), fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / "noise_training_accuracy.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 2 — does the noise change the solution?

This is the test of the hypothesis. **Left:** the angle between the *where* and *whether* decoder axes, so
90 degrees means the two tasks are encoded orthogonally. **Right:** how far those axes sit from the nearest
hidden-unit axis, so 0 degrees means each task has its own unit and 45 degrees means both units carry both
tasks.

If noise drives the shared, rotated solution, the right-hand panel should climb from near 0 at no noise
towards 45 at high noise. A flat line across this range refutes the hypothesis across the full noise
regime rather than only near 0.1.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5.4))
for kind in KINDS:
    c, mk = kind_style[kind]
    m = np.array([np.nanmean(res[kind][ns]["angle"]) for ns in TRAIN_NOISE_LEVELS])
    sd = np.array([np.nanstd(res[kind][ns]["angle"]) for ns in TRAIN_NOISE_LEVELS])
    axs[0].errorbar(TRAIN_NOISE_LEVELS, m, yerr=sd, marker=mk, lw=2.0, capsize=3, color=c, label=kind)
    axs[0].fill_between(TRAIN_NOISE_LEVELS, m-sd, m+sd, color=c, alpha=0.12)
    for key, ls, lab in [("align_where", "-", "where"), ("align_whether", "--", "whether")]:
        mm = np.array([np.nanmean(res[kind][ns][key]) for ns in TRAIN_NOISE_LEVELS])
        axs[1].plot(TRAIN_NOISE_LEVELS, mm, marker=mk, ls=ls, lw=2.0, color=c, label="%s, %s" % (kind, lab))
axs[0].axhline(90, color="green", ls=":", alpha=0.7, label="orthogonal (90\u00b0)")
axs[0].set_ylabel("angle between where and whether axes (deg)"); axs[0].set_ylim(0, 100)
axs[0].set_title("Is the code factorised?")
axs[1].axhline(45, color="purple", ls=":", alpha=0.7, label="fully shared (45\u00b0)")
axs[1].axhline(0,  color="gray", ls=":", alpha=0.7)
axs[1].set_ylabel("alignment to nearest hidden-unit axis (deg)"); axs[1].set_ylim(-2, 50)
axs[1].set_title("One unit per task, or both units shared?")
for ax in axs:
    ax.set_xlabel("training noise (baseline standard deviation)"); ax.grid(alpha=0.3)
    ax.axvline(0.1, color="steelblue", ls="--", alpha=0.5)
    ax.legend(fontsize=8, frameon=False)
fig.suptitle("Does training noise change the solution the network finds?", fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / "noise_training_geometry.png", dpi=150, bbox_inches="tight"); plt.show()

## Figure 3 — example task-space trajectories at low and high noise

A direct visual comparison of the extremes, drawn in the task-aligned space so the two are comparable.

In [ ]:
def subtask_style(s):
    if s == "det_multisensory":   return {"color": "#8c3b00", "ls": "--", "lw": 2.0}
    if s.startswith("det"):       return {"color": "#e0852e", "ls": "-",  "lw": 1.8}
    if "conflict" in s:           return {"color": "#12355b", "ls": "--", "lw": 2.0}
    return {"color": "#5b9bd5", "ls": "-", "lw": 1.8}

show_kind = KINDS[0]
show_levels = [TRAIN_NOISE_LEVELS[0], TRAIN_NOISE_LEVELS[len(TRAIN_NOISE_LEVELS)//2], TRAIN_NOISE_LEVELS[-1]]
fig, axs = plt.subplots(1, len(show_levels), figsize=(5.0*len(show_levels), 4.8))
axs = np.atleast_1d(axs)
for ax, ns in zip(axs, show_levels):
    m, axinfo = example_models[(show_kind, ns)]
    ts = test_by_noise[ns]
    Hs = m.hidden_states(ts["X"])
    Z = np.stack([Hs @ axinfo["w_where"], Hs @ axinfo["w_whether"]], -1)
    for sub in SUBTASKS:
        idx = np.where(ts["types"] == sub)[0]
        tr = Z[idx].mean(0); st = subtask_style(sub)
        ax.plot(tr[:,0], tr[:,1], color=st["color"], ls=st["ls"], lw=st["lw"], alpha=0.9,
                label=sub if ns == show_levels[0] else None)
        ax.scatter(*tr[-1], color=st["color"], s=45, marker="*", edgecolor="k", lw=0.4, zorder=6)
    ax.axhline(0, color="0.7", lw=0.8, ls=":"); ax.axvline(0, color="0.7", lw=0.8, ls=":")
    ax.set_xlabel("z$_{where}$"); ax.set_ylabel("z$_{whether}$")
    ax.set_title("training noise %.2f  (axes %.0f\u00b0)" % (ns, axinfo["angle"]), fontsize=11)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=6, fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.10))
fig.suptitle("Task-aligned trajectories at increasing training noise (%s model)" % show_kind, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig(OUT_DIR / "noise_training_taskspace.png", dpi=150, bbox_inches="tight"); plt.show()

## 7. Summary table and verdict

In [ ]:
print(f"{'kind':>9} {'noise':>7} {'matched':>9} {'clean':>8} {'axes ang':>10} {'align w':>9} {'align d':>9} {'dec acc':>9}")
print("-"*80)
for kind in KINDS:
    for ns in TRAIN_NOISE_LEVELS:
        r = res[kind][ns]
        print(f"{kind:>9} {ns:>7.2f} {np.mean(r['acc_matched']):>9.3f} {np.mean(r['acc_clean']):>8.3f} "
              f"{np.nanmean(r['angle']):>9.1f}\u00b0 {np.nanmean(r['align_where']):>8.1f}\u00b0 "
              f"{np.nanmean(r['align_whether']):>8.1f}\u00b0 {np.nanmean(r['acc_dec_where']):>9.3f}")

k = KINDS[0]
al = np.array([np.nanmean(res[k][ns]["align_where"]) for ns in TRAIN_NOISE_LEVELS])
ang = np.array([np.nanmean(res[k][ns]["angle"]) for ns in TRAIN_NOISE_LEVELS])
ok = ~np.isnan(al)
if ok.sum() > 2:
    corr = np.corrcoef(np.array(TRAIN_NOISE_LEVELS)[ok], al[ok])[0, 1]
    print("\ncorr(training noise, alignment to hidden-unit axes) = %.2f" % corr)
    print("alignment range across the sweep: %.1f to %.1f deg" % (np.nanmin(al), np.nanmax(al)))
    if abs(corr) < 0.4 and (np.nanmax(al) - np.nanmin(al)) < 12:
        print("\nVerdict: alignment does not move systematically with training noise across this range,")
        print("so the noise hypothesis is not supported even when noise is comparable to the signal.")
    elif corr > 0.4:
        print("\nVerdict: alignment increases with training noise, which supports the hypothesis that")
        print("noise pushes the network towards a shared, rotated solution.")
    else:
        print("\nVerdict: alignment decreases with training noise, the opposite of the hypothesis.")
print("\nmean angle between axes across the sweep: %.1f deg (90 = orthogonal)" % np.nanmean(ang))

## Notes

- Two things are separated here that the earlier comparison ran together: **how hard the task is** at a
  given noise level (Figure 1) and **what solution the network settles on** (Figure 2). The hypothesis is
  about the second.
- Decoder accuracy is reported alongside the geometry. At very high noise the network may fail the task,
  in which case the decoders are fitting little and the angles become unreliable, so read the geometry
  panels only where accuracy is still well above chance.
- The original comparison used 0.1 versus 0.0, both of which are small relative to a stimulus intensity of
  1 to 3. If the alignment is flat all the way to noise 2.0, the hypothesis is refuted across the whole
  regime rather than just near zero, which is a much stronger statement for the report.